# Logistics Delay Prediction — SageMaker ML Workflow

**Cloud Computing — Modular Assignment 6**  
**AWS Region:** `eu-north-1`

This notebook documents dataset checks, local inference, the completed SageMaker training job, and the historical real-time endpoint inference workflow. Shipment records are synthetic; weather observations were retrieved using Open-Meteo.

**Important:** The SageMaker endpoint, endpoint configuration and model resource were deleted after successful testing. This notebook does **not** recreate them. Do not run deployment or training again merely to review the assignment. See `docs/evidence/E05`–`E08` for the recorded cloud results.


## 1. Validate the prepared dataset

Validate the 1,000-row processed dataset before reviewing model results.

In [ ]:
from pathlib import Path
import pandas as pd

project = Path.cwd()
if project.name == "notebooks":
    project = project.parent

dataset_path = project / "data" / "processed" / "ml_dataset.csv"
df = pd.read_csv(dataset_path)
print("Dataset shape:", df.shape)
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
assert df.shape == (1000, 16), "Unexpected dataset dimensions"
assert not df.isna().any().any(), "Missing values detected"
assert not df.duplicated().any(), "Duplicate rows detected"
print("Dataset validation: PASSED")
display(df.head())


## 2. Local inference validation

Use the saved local model and the original `deployment/source/inference.py` interface. This runs locally and does not create AWS resources.

In [ ]:
import sys
import json

source_dir = project / "deployment" / "source"
sys.path.insert(0, str(source_dir))
from inference import model_fn, input_fn, predict_fn, output_fn

local_model = model_fn(str(project / "deployment" / "local_model"))
sample = {
    "region": "Riyadh",
    "distance_km": 150,
    "traffic_level": "High",
    "departure_hour": 9,
    "planned_delivery_minutes": 180,
    "temperature_c": 35,
    "humidity_percent": 40,
    "rainfall_mm": 0
}
parsed = input_fn(json.dumps(sample), "application/json")
prediction = predict_fn(parsed, local_model)
print("LOCAL INFERENCE TEST: PASSED")
print(output_fn(prediction, "application/json"))


## 3. SageMaker training configuration (historical)

The successful job used the following settings. This is a record of the configuration, **not a command to launch another job**.

- Region: `eu-north-1`
- S3 input: `s3://logistics-delay-m6-habib/data/ml_dataset.csv`
- Output prefix: `s3://logistics-delay-m6-habib/model-artifacts/`
- Entry point: `deployment/source/train.py`
- Framework: SageMaker Scikit-learn `1.4-2`
- Training instance: `ml.m5.large` (1 instance)
- Execution role: `Logistics-M6-SageMaker-ExecutionRole`
- Successful training job: `logistics-delay-m6-training-2026-10-29-11-22-38-826`

The original notebook contained `SKLearn(...).fit(inputs={"train": S3_TRAINING_DATA}, wait=False, logs=False)`. That command **creates a new billable training job** and has deliberately not been retained as an executable cell. The training implementation remains in `deployment/source/train.py`. Evidence: `docs/evidence/E05-SageMaker-Training-Completed.png`.


## 4. Recorded model evaluation

Evaluation metrics and visualizations are saved locally. The next cell only reads those files.

In [ ]:
metrics_path = project / "models" / "evaluation_metrics.json"
with metrics_path.open(encoding="utf-8") as f:
    metrics = json.load(f)
print(json.dumps(metrics, indent=2))
print("Plots:")
for name in ("actual_vs_predicted.png", "prediction_residuals.png", "weather_model_comparison.png"):
    path = project / "docs" / "plots" / name
    print(name, "PRESENT" if path.is_file() else "MISSING")


## 5. SageMaker deployment and cloud inference (completed; endpoint deleted)

The original notebook included `SKLearnModel.create`, `create_endpoint_config`, `create_endpoint`, `SKLearnModel.deploy`, and `invoke_endpoint` commands, plus troubleshooting checks. The final successful workflow was:

1. Verify the completed SageMaker training job and S3 model artifact.
2. Package the custom `deployment/source/inference.py` functions (`model_fn`, `input_fn`, `predict_fn`, `output_fn`).
3. Create the SageMaker model and endpoint configuration using `ml.m5.large`.
4. Deploy `logistics-delay-m6-endpoint` and confirm `InService`.
5. Invoke the endpoint with a Riyadh shipment JSON payload and record the returned prediction.
6. Inspect CloudWatch endpoint logs.
7. Delete the endpoint, configuration, and SageMaker model after testing.

**Do not rerun deployment:** The endpoint no longer exists. Recreating it would incur AWS charges. The actual deployment and inference results are preserved in the screenshots below, not reproduced as current live results.

- `docs/evidence/E06-SageMaker-Endpoint-InService.png`
- `docs/evidence/E07-SageMaker-Endpoint-Inference.png`
- `docs/evidence/E08-CloudWatch-Endpoint-Logs.png`


## 6. Cloud training artifacts and verification

The successful training job produced `model.tar.gz` and `output.tar.gz`, downloaded to `models/`. The S3 bucket has been retained for the final submission audit.

The following check only inspects the local archives and never contacts AWS.

In [ ]:
import tarfile
for filename in ("model.tar.gz", "output.tar.gz"):
    path = project / "models" / filename
    assert path.is_file(), f"Missing archive: {filename}"
    with tarfile.open(path, "r:gz") as archive:
        members = archive.getnames()
    print(filename, "VALID ARCHIVE", "| entries:", len(members))


## 7. Evidence and reproducibility notes

- `E01` Weather API integration; `E02` shipment dataset; `E03` preprocessing.
- `E04` S3 dataset upload; `E05` completed SageMaker training.
- `E06` endpoint `InService`; `E07` successful endpoint inference; `E08` CloudWatch logs.
- Source code: `scripts/`, `deployment/source/train.py`, `deployment/source/inference.py`.
- Local evaluation: `models/evaluation_metrics.json`, `docs/plots/`.

The cloud execution evidence reflects the earlier completed deployment. The notebook's executable cells are restricted to local validation and artifact inspection, to avoid accidental AWS charges.
